# Workshop, Part 1: Build Your Own Name Generator

**Artificial Intelligence and Machine Learning · Transformers workshop**

In the next 30 minutes you will build a small Transformer, train it on a list of names, and watch it invent new ones. It is the same kind of model as GPT, only about 7,000 times smaller than the language model we will download in Part 3.

**How to work**

- Run the cells from top to bottom with **Shift+Enter**.
- Three cells contain a **TODO**. Replace every `...` with code. The cell after each TODO checks your answer.
- Follow the printed guide: it shows where each TODO sits inside the model.

| Step | What happens | Your job |
|---|---|---|
| 1 | Choose a list of names | change one line |
| 2 | Turn letters into numbers | run |
| 3 | **TODO 1:** attention | 3 lines |
| 4 | **TODO 2:** the mask | 1 line |
| 5 | **TODO 3:** pick the next letter | 2 lines |
| 6 | Build the model | run |
| 7 | Train it | run and watch |
| 8 | Invent names | run and play |
| 9 | Look inside the model | run |
| 10 | Save the model for your app | run |

## 1. Choose your list

Change `LIST` to the list you want and write your name in `AUTHOR`, between the quotes. Both appear in your app later.

| `LIST` | What it contains |
|---|---|
| `"lithuanian_names"` | 518 Lithuanian first names |
| `"english_names"` | 3,798 English first names |
| `"italian_towns"` | 3,999 Italian towns (they sound like pasta) |
| `"swedish_towns"` | 1,168 Swedish towns (they sound like furniture) |
| `"japanese_cities"` | 1,826 Japanese cities |

You can also bring your own list: a text file in the `data` folder with one word per line and at least 300 words.

In [ ]:
import copy
import math
import random
import time
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

LIST = "lithuanian_names"       # <-- choose your list
AUTHOR = "Your Name"            # <-- write your name between the quotes

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)        # one processor thread is enough for a model this small
if "CHECKS" not in globals():
    CHECKS = {}                 # filled in by the three check cells

list_file = Path(f"data/{LIST}.txt")
if not list_file.exists():
    raise FileNotFoundError(f"There is no file data/{LIST}.txt. Write LIST without '.txt', and open "
                            "the whole workshop folder in VS Code (File > Open Folder).")
try:
    text = list_file.read_text(encoding="utf-8-sig")
except UnicodeDecodeError:
    raise ValueError(f"data/{LIST}.txt is not saved as UTF-8. Save it again with the encoding UTF-8.") from None

# Lowercase letters only, every word once.
words = list(dict.fromkeys(unicodedata.normalize("NFC", text).lower().split()))
if len(words) < 300:
    raise ValueError(f"The list has only {len(words)} words. It needs at least 300.")

print(f"List: {LIST}  ({len(words):,} words)")
print("Examples:", ", ".join(random.sample(words, 8)))

## 2. Letters become numbers

A model needs numbers, so every letter gets a number. One extra symbol, the full stop `.`, marks the **start and the end** of a word.

The model's task is always the same: **look at the letters so far and predict the next one.** So the targets are the inputs moved one step to the left.

In [ ]:
# The alphabet of your list. Position 0 is the start/end symbol.
chars = ["."] + sorted(set("".join(words)))
char_to_id = {c: i for i, c in enumerate(chars)}
VOCAB_SIZE = len(chars)
BLOCK_SIZE = max(len(w) for w in words) + 1      # longest word plus the start symbol


def encode(word_list):
    # One row per word: inputs X and targets Y. Unused positions get target -1 (ignored).
    X = torch.zeros(len(word_list), BLOCK_SIZE, dtype=torch.long)
    Y = torch.full((len(word_list), BLOCK_SIZE), -1)
    for row, word in enumerate(word_list):
        ids = [0] + [char_to_id[c] for c in word] + [0]
        X[row, :len(ids) - 1] = torch.tensor(ids[:-1])
        Y[row, :len(ids) - 1] = torch.tensor(ids[1:])
    return X, Y


# 90% of the words for training, 10% to check the model on words it has not seen.
shuffled = words.copy()
random.Random(SEED).shuffle(shuffled)
n_val = max(50, len(shuffled) // 10)
X_train, Y_train = encode(shuffled[n_val:])
X_val, Y_val = encode(shuffled[:n_val])

print(f"Alphabet ({VOCAB_SIZE} symbols): {' '.join(chars)}")
print(f"Longest word: {BLOCK_SIZE - 1} letters")
example = shuffled[n_val]
print(f"\nExample word: {example}")
print("The model sees   ->  and must predict")
for i in range(len(example) + 1):
    seen = "." + example[:i]
    target = (example + ".")[i]
    print(f"  {seen:<{BLOCK_SIZE}} ->  {target}")

## 3. TODO 1: attention

This is the heart of the Transformer, the formula from the lecture:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d}}\right)V$$

In words: **match** every query with every key, turn the scores into **shares** with a softmax, and **blend** the values.

Useful pieces:

- `a @ b` multiplies two tensors (dot products of rows with columns);
- `k.transpose(-2, -1)` is $K^\top$;
- `math.sqrt(d)` is $\sqrt{d}$;
- `torch.softmax(x, dim=-1)` is a softmax over the last axis.

In [ ]:
def attention(q, k, v, mask):
    # q, k, v: (..., positions, d).  mask: True where a position may be looked at.
    d = q.shape[-1]        # the length of the vectors this function receives

    # TODO 1a (match): the dot product of every query with every key, divided by sqrt(d)
    scores = ...

    scores = scores.masked_fill(~mask, float("-inf"))   # given: hide what must not be seen

    # TODO 1b (share): a softmax over the last axis turns scores into weights
    weights = ...

    # TODO 1c (blend): the weights times the values
    output = ...

    return output, weights

In [ ]:
# CHECK 1: the worked example from the lecture (one query, three keys, the answer is 14.25),
# and then the same function the way the model uses it (many words, two heads).
def works_for_many_words():
    generator = torch.Generator().manual_seed(0)
    q, k, v = (torch.randn(3, 2, 5, 9, generator=generator) for _ in range(3))
    mask = torch.tril(torch.ones(5, 5, dtype=torch.bool))
    scores = (q @ k.transpose(-2, -1) / 3).masked_fill(~mask, float("-inf"))
    expected = torch.softmax(scores, dim=-1) @ v
    try:
        output, _ = attention(q, k, v, mask)
        return output.shape == expected.shape and torch.allclose(output, expected, atol=1e-5)
    except Exception:
        return False


def check_attention():
    q = torch.tensor([[1.0, 1.0, 1.0, 1.0]])
    k = torch.tensor([[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 0.0, 0.0], [1.0, -1.0, 1.0, -1.0]])
    v = torch.tensor([[10.0], [20.0], [30.0]])
    try:
        output, weights = attention(q, k, v, torch.ones(1, 3, dtype=torch.bool))
        if not torch.is_tensor(output) or not torch.is_tensor(weights):
            print("NOT YET: a line still contains ... ; replace every ... with code.")
        elif output.numel() != 1:
            print("NOT YET: the output has the wrong shape. The last line is: output = weights @ v")
        elif abs(weights.sum().item() - 1) > 1e-4:
            print("NOT YET: the weights must sum to 1. Did you use torch.softmax(scores, dim=-1)?")
        elif abs(output.item() - 14.25) > 0.01:
            print(f"NOT YET: the output is {output.item():.2f}, it should be 14.25.")
            print("         Check the scores: q @ k.transpose(-2, -1) / math.sqrt(d)")
        elif not works_for_many_words():
            print("NOT YET: right for one word, but the model sends many words and two heads at once.")
            print("         Use k.transpose(-2, -1) (not k.T), math.sqrt(d) (not a fixed number),")
            print("         and dim=-1 (not dim=1).")
        else:
            print("CORRECT. Weights:", [round(w, 2) for w in weights[0].tolist()],
                  "| output:", round(output.item(), 2))
            return True
    except Exception as error:
        message = str(error)
        if "ellipsis" in message:
            print("NOT YET: a line still contains ... ; replace every ... with code.")
        else:
            print("NOT YET:", type(error).__name__, "-", message)
            if "shape" in message or "size" in message:
                print("         Check the shapes: the keys need .transpose(-2, -1).")
    return False


CHECKS[1] = check_attention()

## 4. TODO 2: the mask

When the model predicts the next letter, it may look only at the letters **so far**, never ahead. The mask is a table of `True` (may look) and `False` (may not): row = the position that looks, column = the position it looks at.

For four positions it should be:

```
True  False False False
True  True  False False
True  True  True  False
True  True  True  True
```

Your function must work for **any** size, so use the argument `size`.

Useful pieces: `torch.ones(size, size, dtype=torch.bool)` is a table full of `True`, and `torch.tril(table)` keeps the lower-left triangle and sets the rest to `False`.

In [ ]:
def causal_mask(size):
    # TODO 2: a size-by-size table, True on and below the diagonal, False above it
    mask = ...
    return mask

In [ ]:
# CHECK 2
def check_mask():
    try:
        mask = causal_mask(4)
        expected = torch.tensor([[1, 0, 0, 0], [1, 1, 0, 0], [1, 1, 1, 0], [1, 1, 1, 1]], dtype=torch.bool)
        if mask is ...:
            print("NOT YET: replace the ... with code.")
        elif not torch.is_tensor(mask):
            print("NOT YET: the result must be a PyTorch tensor. Use torch.tril(torch.ones(...)).")
        elif mask.dtype != torch.bool:
            print("NOT YET: the table must contain True/False. Add dtype=torch.bool inside torch.ones(...).")
        elif mask.shape != (4, 4) or not torch.equal(mask, expected):
            print("NOT YET: this is your table for size 4:")
            print(mask.int())
            print("         It should be 1 on and below the diagonal, 0 above. Try torch.tril(...).")
        elif not torch.equal(causal_mask(6), torch.tril(torch.ones(6, 6, dtype=torch.bool))):
            print("NOT YET: right for size 4 only. Use the argument: torch.ones(size, size, dtype=torch.bool).")
        else:
            print("CORRECT. Each row may look at itself and at everything before it:")
            print(mask.int())
            return True
    except Exception as error:
        print("NOT YET:", type(error).__name__, "-", error)
    return False


CHECKS[2] = check_mask()

## 5. TODO 3: pick the next letter

The model gives a **score** for every letter. To choose one:

1. divide the scores by the **temperature** and apply a softmax, which gives probabilities;
2. draw one letter at random with those probabilities.

A low temperature (0.5) makes the model careful: it usually takes its favourite letter. A high temperature (1.5) makes it adventurous.

Useful pieces: `torch.softmax(x, dim=-1)` and `torch.multinomial(probabilities, 1)`, which draws one position at random.

In [ ]:
def pick_next(scores, temperature=1.0):
    # scores: one number per symbol of the alphabet.
    # TODO 3a: probabilities = softmax of (scores divided by temperature)
    probabilities = ...

    # TODO 3b: draw one symbol at random with these probabilities
    next_id = ...

    return int(next_id)

In [ ]:
# CHECK 3: three symbols with scores 0, 0 and 3. How often does the favourite (the third) win?
def check_pick_next():
    try:
        torch.manual_seed(0)
        scores = torch.tensor([0.0, 0.0, 3.0])
        share = {}
        for temperature, draws in [(0.1, 200), (1.0, 2000), (2.0, 2000), (50.0, 2000)]:
            picks = [pick_next(scores, temperature=temperature) for _ in range(draws)]
            share[temperature] = picks.count(2) / draws
        hint = "         Line 1: torch.softmax(scores / temperature, dim=-1). Line 2: torch.multinomial(probabilities, 1)."
        if share[0.1] < 1:
            print(f"NOT YET: at temperature 0.1 the favourite should win every time, not {share[0.1]:.0%}.")
            print(hint)
        elif not 0.86 < share[1.0] < 0.96:
            print(f"NOT YET: at temperature 1 the favourite should win about 91% of the time, not {share[1.0]:.0%}.")
            print(hint)
        elif not 0.63 < share[2.0] < 0.75:
            print(f"NOT YET: at temperature 2 the favourite should win about 69% of the time, not {share[2.0]:.0%}.")
            print(hint)
        elif not 0.28 < share[50.0] < 0.42:
            print(f"NOT YET: at temperature 50 all symbols should be almost equally likely, not {share[50.0]:.0%}.")
            print(hint)
        else:
            print("CORRECT. How often the favourite symbol wins:")
            print("  " + "   ".join(f"temperature {t:g}: {s:.0%}" for t, s in share.items()))
            return True
    except Exception as error:
        if "ellipsis" in str(error):
            print("NOT YET: a line still contains ... ; replace every ... with code.")
        else:
            print("NOT YET:", type(error).__name__, "-", error)
            print("         Use torch.softmax(..., dim=-1) and torch.multinomial(..., 1).")
    return False


CHECKS[3] = check_pick_next()

## 6. Build the model

Now we assemble the Transformer from your pieces. You do not need to change anything here; read the comments and find your three functions.

```
letters so far ->  letter embedding + position embedding
               ->  Block 1:  self-attention (your attention + your mask)  ->  small MLP
               ->  Block 2:  self-attention (your attention + your mask)  ->  small MLP
               ->  a score for every letter  ->  your pick_next
```

**Write the number of parameters into your notes on page 1 of the guide.**

In [ ]:
missing = [str(number) for number in (1, 2, 3) if not CHECKS.get(number)]
if len(missing) == 1:
    raise RuntimeError(f"Not ready: check {missing[0]} has not said CORRECT yet. "
                       "Finish that TODO, run it, and run its check cell.")
if missing:
    raise RuntimeError(f"Not ready: checks {', '.join(missing[:-1])} and {missing[-1]} have not said CORRECT yet. "
                       "Finish each TODO, run it, and run its check cell.")


class SelfAttention(nn.Module):
    # Multi-head self-attention: every letter looks at the letters before it.

    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.to_q = nn.Linear(d, d, bias=False)      # makes the queries
        self.to_k = nn.Linear(d, d, bias=False)      # makes the keys
        self.to_v = nn.Linear(d, d, bias=False)      # makes the values
        self.mix = nn.Linear(d, d, bias=False)       # mixes the heads at the end

    def split(self, x):
        # (words, positions, d) -> (words, heads, positions, d / heads)
        count, positions, d = x.shape
        return x.reshape(count, positions, self.heads, d // self.heads).transpose(1, 2)

    def forward(self, x, mask):
        q, k, v = self.split(self.to_q(x)), self.split(self.to_k(x)), self.split(self.to_v(x))
        out, self.weights = attention(q, k, v, mask)             # <-- YOUR attention
        count, _, positions, _ = out.shape
        return self.mix(out.transpose(1, 2).reshape(count, positions, -1))


class Block(nn.Module):
    # One Transformer block: attention, add & norm, small MLP, add & norm.

    def __init__(self, d, heads, dropout):
        super().__init__()
        self.attention = SelfAttention(d, heads)
        self.norm1 = nn.LayerNorm(d)
        self.mlp = nn.Sequential(nn.Linear(d, 2 * d), nn.ReLU(), nn.Linear(2 * d, d))
        self.norm2 = nn.LayerNorm(d)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        x = self.norm1(x + self.dropout(self.attention(x, mask)))
        return self.norm2(x + self.dropout(self.mlp(x)))


class NameGPT(nn.Module):
    # Letters in, a score for every possible next letter out.

    def __init__(self, vocab_size, block_size, d=32, heads=2, blocks=2, dropout=0.2):
        super().__init__()
        self.settings = {"vocab_size": vocab_size, "block_size": block_size,
                         "d": d, "heads": heads, "blocks": blocks}
        self.block_size = block_size
        self.letter_embedding = nn.Embedding(vocab_size, d)      # what the letter is
        self.position_embedding = nn.Embedding(block_size, d)    # where it stands
        self.blocks = nn.ModuleList([Block(d, heads, dropout) for _ in range(blocks)])
        self.to_scores = nn.Linear(d, vocab_size)

    def forward(self, ids):
        positions = ids.shape[1]
        x = self.letter_embedding(ids) + self.position_embedding(torch.arange(positions))
        mask = causal_mask(positions)                            # <-- YOUR mask
        for block in self.blocks:
            x = block(x, mask)
        return self.to_scores(x)


def generate(model, start="", temperature=1.0):
    # Invent one word, letter by letter, until the model says "stop" (symbol 0).
    model.eval()
    ids = [0] + [char_to_id[c] for c in start.lower() if c in char_to_id and c != "."]
    with torch.no_grad():
        while len(ids) < model.block_size:
            scores = model(torch.tensor([ids]))[0, -1]
            next_id = pick_next(scores, temperature)             # <-- YOUR pick_next
            if next_id == 0 and len(ids) > 1:                    # "stop", but never an empty word
                break
            if next_id != 0:
                ids.append(next_id)
    return "".join(chars[i] for i in ids[1:])


SETTINGS = {"d": 32, "heads": 2, "blocks": 2}       # the size of your model


def new_model():
    # A fresh model that knows nothing yet.
    torch.manual_seed(SEED)
    return NameGPT(VOCAB_SIZE, BLOCK_SIZE, **SETTINGS)


model = new_model()
history = None                                       # filled in by the training in Step 7
num_parameters = sum(p.numel() for p in model.parameters())
print(f"Your model has {num_parameters:,} trainable parameters.")
print("\nNames BEFORE training (the model knows nothing yet):")
print("  " + ", ".join(generate(model) for _ in range(8)))

## 7. Train the model

Training repeats one small step 2,000 times:

1. take 64 random words;
2. let the model predict every next letter;
3. measure how wrong it was (the **loss**);
4. change every parameter a little so that it is less wrong next time.

Every 400 steps the model shows you three names it invents at that moment. **Watch them improve.** This takes 10 to 40 seconds. **Write the seconds into your notes on page 1 of the guide.**

In the figure, the orange line is measured on words the model has never seen. When it starts to rise, the model has begun to **memorise** your list instead of learning its style. We keep the model from the lowest point of the orange line.

In [ ]:
def loss_on(model, X, Y):
    scores = model(X)
    return F.cross_entropy(scores.reshape(-1, VOCAB_SIZE), Y.reshape(-1), ignore_index=-1)


def train(model, steps=2000, batch_size=64, learning_rate=0.003):
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    start = time.time()
    model.eval()
    with torch.no_grad():                                         # the loss before any training
        first = loss_on(model, X_val, Y_val).item()
    history = {"step": [0], "train": [first], "val": [first]}
    best_val, best_state, best_step = first, copy.deepcopy(model.state_dict()), 0
    recent = []
    for step in range(1, steps + 1):
        model.train()
        rows = torch.randint(0, len(X_train), (batch_size,))     # 1. random words
        loss = loss_on(model, X_train[rows], Y_train[rows])       # 2.-3. predict and measure
        optimizer.zero_grad()
        loss.backward()                                           # 4. how to change
        optimizer.step()                                          #    change a little
        recent.append(loss.item())

        if step % 100 == 0 or step == steps:                      # check on unseen words
            model.eval()
            with torch.no_grad():
                val = loss_on(model, X_val, Y_val).item()
            history["step"].append(step)
            history["train"].append(sum(recent) / len(recent))
            history["val"].append(val)
            recent = []
            if val < best_val:                                    # keep the best model so far
                best_val, best_state, best_step = val, copy.deepcopy(model.state_dict()), step
        if step % 400 == 0:
            samples = ", ".join(generate(model) for _ in range(3))
            print(f"step {step:4d} | loss {history['train'][-1]:.2f} | {samples}")

    model.load_state_dict(best_state)                             # go back to the best model
    history["best_step"] = best_step
    print(f"\nDone in {time.time() - start:.0f} seconds.")
    print(f"Kept the model from step {best_step}: the one that was best on words it had never seen.")
    return history


model = new_model()                    # always start from an untrained model
history = train(model)

plt.figure(figsize=(6.5, 3.2))
plt.plot(history["step"], history["train"], label="words used for training")
plt.plot(history["step"], history["val"], label="words never seen")
plt.axvline(history["best_step"], color="gray", linestyle="--", label="model we keep")
plt.xlabel("Training step")
plt.ylabel("Loss (lower is better)")
plt.title("How wrong the model is")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Invent names

A star marks a name that is **not** in your list: the model invented it.

Then compare three **temperatures**, and try the **first letters**. Your app will have both controls.

In [ ]:
known = set(words)


def show(names):
    return ", ".join(name + ("" if name in known else " *") for name in names)


torch.manual_seed(SEED)
print("Twenty names from your model (* = invented):\n")
names = [generate(model, temperature=0.8) for _ in range(20)]
print(show(names[:10]))
print(show(names[10:]))
invented = sum(name not in known for name in names)
print(f"\n{invented} of 20 are not in the list.")

print("\nSame model, different temperature:")
for temperature in [0.5, 1.0, 1.5]:
    print(f"  {temperature}: " + show([generate(model, temperature=temperature) for _ in range(6)]))

first_letters = words[0][:2]
print(f"\nNames that start with '{first_letters}':")
print("  " + show([generate(model, start=first_letters, temperature=0.8) for _ in range(6)]))

## 9. Look inside your model

The model has two blocks with two attention heads each. The picture shows where each head of the **last block** looks while the model spells one invented name.

Read it row by row: the row is the letter that is looking, the columns are the letters it may look at, and a darker cell means more attention. The empty upper triangle is your mask.

In [ ]:
torch.manual_seed(SEED + 1)
name = generate(model, temperature=0.8)
ids = [0] + [char_to_id[c] for c in name]
model.eval()
with torch.no_grad():
    model(torch.tensor([ids]))
weights = model.blocks[-1].attention.weights[0]          # (heads, positions, positions)

labels = ["."] + list(name)
fig, axes = plt.subplots(1, weights.shape[0], figsize=(4.2 * weights.shape[0], 3.9), squeeze=False)
for head, ax in enumerate(axes[0]):
    ax.imshow(weights[head].tolist(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("letter it looks at")
    ax.set_title(f"Head {head + 1}")
axes[0][0].set_ylabel("letter that is looking")
fig.suptitle(f"Attention in the last block while spelling '{name}'")
plt.tight_layout()
plt.show()

## 10. Save your model for the app

This cell writes one small file, `app/model.pt`. It contains everything the app needs: the trained parameters, your alphabet, and your list.

**Next:** go to **Part 2** of the printed guide and put your model online.

In [ ]:
if history is None:
    raise RuntimeError("Train the model first: run Step 7.")
if model.settings["vocab_size"] != VOCAB_SIZE or model.settings["block_size"] != BLOCK_SIZE:
    raise RuntimeError("The model was built for another list. Choose Run All to build and train it again.")

Path("app").mkdir(exist_ok=True)
torch.save({
    "state_dict": model.state_dict(),
    "settings": model.settings,
    "chars": chars,
    "words": words,
    "list_name": LIST,
    "author": AUTHOR,
    "parameters": num_parameters,
}, "app/model.pt")

size_kb = Path("app/model.pt").stat().st_size / 1024
print(f"Saved app/model.pt ({size_kb:.0f} kB)")
print(f"List: {LIST} | author: {AUTHOR} | parameters: {num_parameters:,}")
if AUTHOR == "Your Name":
    print("\nYour app will say 'Built by Your Name'. Write your own name in Step 1 and choose Run All.")
print("\nNext: Part 2 of the printed guide.")

## If you have time

- Change `LIST` in Step 1, choose **Run All**, and compare the names.
- In Step 6, change `SETTINGS` to `{"d": 64, "heads": 2, "blocks": 2}` and choose **Run All**: a model with almost four times as many parameters.
- Make your own list: put a text file with one word per line (at least 300 words, saved as UTF-8) into the `data` folder and use its name without `.txt` as `LIST`.